# 🏗️ Medallion Architecture — TriageAI
Pipeline de dados: Bronze → Silver → Gold

Este notebook pode ser executado de duas formas:
- **No Google Colab** (após clonar o repositório)
- **Localmente** com Jupyter (dentro da pasta do projeto)

Os arquivos serão salvos automaticamente nas pastas `data/bronze/`, `data/silver/` e `data/gold/` do repositório.

## 0. Setup — Clonar o repositório (apenas no Colab)

In [ ]:
import os
import sys

# Detecta se está rodando no Google Colab
IN_COLAB = 'google.colab' in sys.modules

REPO_URL  = "https://github.com/lucyexz/TriageAI.git"
REPO_NAME = "TriageAI"

if IN_COLAB:
    if not os.path.exists(REPO_NAME):
        print("Clonando repositório...")
        os.system(f"git clone {REPO_URL}")
    os.chdir(REPO_NAME)
    print(f"✅ Diretório: {os.getcwd()}")
else:
    # Execução local: sobe até a raiz do projeto se estiver dentro de /notebooks
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        os.chdir('..')
    print(f"✅ Diretório: {os.getcwd()}")

## 1. Configuração dos caminhos das camadas Medallion

In [ ]:
import pandas as pd
import kagglehub

# Raiz do projeto (onde está o notebook ou o repo clonado)
BASE_DIR = os.getcwd()

# Caminhos das camadas — alinhados com a estrutura do repositório
BRONZE_PATH = os.path.join(BASE_DIR, "data", "bronze")
SILVER_PATH = os.path.join(BASE_DIR, "data", "silver")
GOLD_PATH   = os.path.join(BASE_DIR, "data", "gold")

# Cria as pastas se ainda não existirem
for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    os.makedirs(path, exist_ok=True)

print("📁 Estrutura de pastas:")
print(f"  Bronze : {BRONZE_PATH}")
print(f"  Silver : {SILVER_PATH}")
print(f"  Gold   : {GOLD_PATH}")

## 2. Bronze — Ingestão dos dados brutos

In [ ]:
# Download do dataset via kagglehub
dataset_path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("📦 Dataset baixado em:", dataset_path)

In [ ]:
# Localiza o CSV dentro da pasta baixada pelo kagglehub
csv_filename = "Final_Augmented_dataset_Diseases_and_Symptoms.csv"
source_csv   = os.path.join(dataset_path, csv_filename)

# Lê o dataset bruto
df = pd.read_csv(source_csv)

# Salva na camada Bronze
bronze_file = os.path.join(BRONZE_PATH, "Bronze.csv")
df.to_csv(bronze_file, index=False)

print(f"✅ Bronze salvo em: {bronze_file}")
print(f"   Shape: {df.shape}")
df.head()

## 3. Silver — Limpeza e estruturação

In [ ]:
col_doenca   = 'diseases'
sintomas_cols = df.columns.drop(col_doenca)

# Extrai lista de sintomas presentes (valor == 1) para cada linha
def extrair_sintomas_linha(row):
    return [col for col in sintomas_cols if row[col] == 1]

df['sintomas'] = df.apply(extrair_sintomas_linha, axis=1)
df.head()

In [ ]:
# Agrupa todos os sintomas únicos por doença
def unir_sintomas(grupo):
    sintomas = set()
    for lista in grupo:
        sintomas.update(lista)
    return list(sintomas)

df_silver = (
    df.groupby(col_doenca)['sintomas']
      .apply(unir_sintomas)
      .reset_index()
)

# Normaliza nomes: lowercase + remove underscores dos sintomas
df_silver[col_doenca] = df_silver[col_doenca].str.lower().str.strip()
df_silver['sintomas'] = df_silver['sintomas'].apply(
    lambda lista: [s.replace('_', ' ').lower() for s in lista]
)

# Salva na camada Silver
silver_file = os.path.join(SILVER_PATH, "Silver.csv")
df_silver.to_csv(silver_file, index=False)

print(f"✅ Silver salvo em: {silver_file}")
print(f"   Shape: {df_silver.shape}")
df_silver.head(10)

## 4. Gold — Textos prontos para o RAG

In [ ]:
# Gera texto descritivo por doença (formato RAG)
def gerar_texto(row):
    sintomas = ', '.join(row['sintomas'])
    return (
        f"{row['diseases'].capitalize()} is a condition that may present symptoms such as {sintomas}. "
        f"Individuals experiencing these symptoms may be associated with this condition. "
        f"This information is for educational purposes only and does not replace professional medical diagnosis."
    )

df_gold = df_silver.copy()
df_gold['texto'] = df_gold.apply(gerar_texto, axis=1)

# Salva na camada Gold
gold_file = os.path.join(GOLD_PATH, "Gold.csv")
df_gold.to_csv(gold_file, index=False)

print(f"✅ Gold salvo em: {gold_file}")
print(f"   Shape: {df_gold.shape}")
df_gold.head(10)

## 5. (Opcional) Enriquecimento — múltiplas variações de texto para o RAG

In [ ]:
def generate_multiple_texts(row):
    symptoms = ', '.join(row['sintomas'])
    disease  = row['diseases']
    return [
        f"{disease.capitalize()} may cause symptoms such as {symptoms}.",
        f"Patients experiencing {symptoms} may be associated with {disease}.",
        f"Common symptoms of {disease} include {symptoms}.",
        f"{disease.capitalize()} is often linked to symptoms like {symptoms}.",
        f"If a person presents with {symptoms}, {disease} could be a possible related condition."
    ]

# Expande cada doença em 5 linhas de texto
df_gold_enriched = df_gold.copy()
df_gold_enriched['textos_multiplos'] = df_gold_enriched.apply(generate_multiple_texts, axis=1)

# Explode para uma linha por texto (ideal para embeddings)
df_gold_exploded = df_gold_enriched.explode('textos_multiplos').reset_index(drop=True)

# Salva versão enriquecida também na camada Gold
gold_enriched_file = os.path.join(GOLD_PATH, "Gold_enriched.csv")
df_gold_exploded[['diseases', 'textos_multiplos']].to_csv(gold_enriched_file, index=False)

print(f"✅ Gold enriquecido salvo em: {gold_enriched_file}")
print(f"   Shape: {df_gold_exploded.shape}")
df_gold_exploded[['diseases', 'textos_multiplos']].head(10)

## ✅ Resumo do pipeline

In [ ]:
print("=" * 50)
print("Pipeline Medallion concluído!")
print("=" * 50)
for label, fpath in [("Bronze", bronze_file), ("Silver", silver_file), ("Gold", gold_file), ("Gold Enriched", gold_enriched_file)]:
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {label:15s}: {fpath}  ({size_kb:.1f} KB)")